In [ ]:
import numpy as np
import pandas as pd
import math

In [ ]:
df = pd.read_csv("/content/housing.csv")
df.head(30)
df.replace({'NaN': 537}, inplace= True)

In [ ]:
y = df['median_house_value'].copy()
y.head(10)

0    452600.0
1    358500.0
2    352100.0
3    341300.0
4    342200.0
5    269700.0
6    299200.0
7    241400.0
8    226700.0
9    261100.0
Name: median_house_value, dtype: float64

In [ ]:
x = df.drop(['median_house_value'],axis= 1)
x.drop(['ocean_proximity'], axis= 1, inplace= True)
x

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462
...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672


In [ ]:
x['xo'] = 1
x
mean_x = x.mean()
mean_x

longitude             -119.569704
latitude                35.631861
housing_median_age      28.639486
total_rooms           2635.763081
total_bedrooms         537.870553
population            1425.476744
households             499.539680
median_income            3.870671
xo                       1.000000
dtype: float64

In [ ]:
m,n = x.shape
w = np.random.rand(n)
w
from sklearn.model_selection import train_test_split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
# reseting the indexing
x_train = x_train.reset_index(drop=True)
x_test = x_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)


In [ ]:
x_train.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,xo
0,-117.03,32.71,33.0,3126.0,627.0,2300.0,623.0,3.2596,1
1,-118.16,33.77,49.0,3382.0,787.0,1314.0,756.0,3.8125,1
2,-120.48,34.66,4.0,1897.0,331.0,915.0,336.0,4.1563,1
3,-117.11,32.69,36.0,1421.0,367.0,1418.0,355.0,1.9425,1
4,-119.80,36.78,43.0,2382.0,431.0,874.0,380.0,3.5542,1


In [ ]:
x_test.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,xo
0,-119.01,36.06,25.0,1505.0,NaN,1392.0,359.0,1.6812,1
1,-119.46,35.14,30.0,2943.0,NaN,1565.0,584.0,2.5313,1
2,-122.44,37.80,52.0,3830.0,NaN,1310.0,963.0,3.4801,1
3,-118.72,34.28,17.0,3051.0,NaN,1705.0,495.0,5.7376,1
4,-121.93,36.62,34.0,2351.0,NaN,1063.0,428.0,3.7250,1


In [ ]:
def hypothesis(X,w):
  return np.dot(X,w)



In [ ]:
z = hypothesis(x_train.loc[0].values,w)
z

3140.777480327058

In [ ]:
def cost_function(x_train,y_train,w):
  m,n = x_train.shape
  cost = 0
  for i in range(m):
    cost += (hypothesis(x_train.loc[i].values,w) - y_train[i])**2
  return cost



In [ ]:
def derivative(x_train,y_train,w):
  m,n = x_train.shape
  dJ_dw = np.zeros(n)
  for i in range(m):
    error = hypothesis(x_train.loc[i].values,w) - y[i]
    for j in range(n):
      dJ_dw[j] = error*x_train.iloc[i,j]
  return dJ_dw/m



In [ ]:
def batch_gradient(x,y,w,alpha,no_iters):
  cost_hist = []
  for i in range(no_iters):
    dJ_dw = derivative(x,y,w)
    w = w - alpha*dJ_dw
    if i < 10000:
      cost_hist.append(cost_function(x,y,w))
    if i%math.ceil(no_iters/10)==0:
      print(f"iteration: {i:4d} : Cost {cost_hist[-1]:8.2f}")
  return w,cost_hist



In [ ]:
def shocastic_gradient(x,y,w,alpha,no_iters):
  m,n = x.shape
  cost_hist = []
  for i in range(no_iters):
    for j in range(n):
      for k in range(m):
        error = (y[k] - hypothesis(x.loc[k].values,w))*x.iloc[k,j]
        w[j] = w[j] + (alpha*error)
    if i < 10000:
      cost_hist.append(cost_function(x,y,w))
    #if i%math.ceil(no_iters/10)==0:
      print(f"iteration: {i:4d} : Cost {cost_hist[-1]:8.2f}")
  return w,cost_hist

In [ ]:
alpha = 0.000000005
no_iters = 10
w = np.zeros(9)
w_final, cost_final = shocastic_gradient(x_train,y_train,w,alpha,no_iters)
w_final

iteration:    0 : Cost 221996909039127.81
iteration:    1 : Cost 218804423769459.38
iteration:    2 : Cost 219113427513905.31
iteration:    3 : Cost 222050118892162.06
iteration:    4 : Cost 227173625518152.47
iteration:    5 : Cost 234251996449371.00
iteration:    6 : Cost 243155186626914.62
iteration:    7 : Cost 253804698061391.31


KeyboardInterrupt: 

In [ ]:
alpha = 0.0005
no_iters = 100
w = [0.12040087, 0.82589774, 0.19043492, 2.70207515, 1.61707568,
       1.94455175, 1.21615926, 0.84019012, 0.34897282]
w_final, cost_final = batch_gradient(x_train,y_train,w,alpha,no_iters)
w_final

In [ ]:
y2 = hypothesis(x_test.loc[0].values,w_final)
y2, y_test[0]


In [ ]:
y1 = []
m,n = x_train.shape
for i in range(m):
  y1.append(np.round(hypothesis(x_train.loc[i].values,w_final)))



In [ ]:
data = {"column1": y_train, "column2":y1,"column3": y_train - y1}
df = pd.DataFrame(data)
df.head(50)